In [1]:
!pip install biopython chembl-webresource-client rdkit
!pip install torch-geometric
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.3.0+cu121.html
!pip install biopython
!pip install chembl-webresource-client
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.9 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.3.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 77.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 70.4 MB/s eta 0:00:00


In [3]:

from Bio import Entrez, SeqIO
import time
from chembl_webresource_client.new_client import new_client
import pandas as pd


# ─────────────────────────────────────────────────────────────────────────────
# HÜCRE 3 — Veri Toplama: ChEMBL + NCBI
# ─────────────────────────────────────────────────────────────────────────────

target_genes = ['ABL1', 'FLT3', 'BTK', 'KIT', 'BCL2', 'JAK2']

ncbi_protein_ids = {
    'ABL1': 'NP_005148',
    'FLT3': 'NP_004110',
    'BTK':  'NP_000052',
    'KIT':  'NP_000213',
    'BCL2': 'NP_000624',
    'JAK2': 'NP_004963'
}

Entrez.email = "240542017@firat.edu.com"

all_smiles_data = []
print("İlaç verileri ChEMBL üzerinden çekiliyor...")

for gene in target_genes:
    print(f"  -> {gene} taranıyor...")
    target     = new_client.target
    target_query = target.filter(target_synonym__icontains=gene).filter(target_organism__icontains='Homo sapiens')

    if not target_query:
        print(f"     UYARI: {gene} ChEMBL'de bulunamadı.")
        continue

    target_id = target_query[0]['target_chembl_id']
    activity  = new_client.activity
    res       = activity.filter(target_chembl_id=target_id).filter(standard_type="IC50")

    count = 0
    for item in res:
        if item['canonical_smiles']:
            all_smiles_data.append({
                'Gene':   gene,
                'SMILES': item['canonical_smiles'],
                'IC50':   item['value']
            })
            count += 1
    print(f"     {count} molekül eklendi.")

df_all = pd.DataFrame(all_smiles_data)
print(f"\nTOPLAM HAM VERİ: {len(df_all)} satır.")

print("\nProtein dizileri NCBI üzerinden çekiliyor...")
with open("leukemia_targets_combined.fasta", "w") as fasta_file:
    for gene, ref_id in ncbi_protein_ids.items():
        try:
            handle   = Entrez.efetch(db="protein", id=ref_id, rettype="fasta", retmode="text")
            seq_data = handle.read()
            fasta_file.write(seq_data)
            print(f"  -> {gene} ({ref_id}) eklendi.")
        except Exception as e:
            print(f"  -> HATA: {gene} çekilemedi. {e}")

print("Tüm proteinler 'leukemia_targets_combined.fasta' dosyasına kaydedildi.")


# ─────────────────────────────────────────────────────────────────────────────
# HÜCRE 4 — RDKit ile Temizleme & Kanonikleştirme
# ─────────────────────────────────────────────────────────────────────────────

from rdkit import Chem

def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            return Chem.MolToSmiles(mol, canonical=True)
        return None
    except:
        return None

print("Veri temizleniyor ve kanonikleştiriliyor...")
df_all['Canonical_SMILES'] = df_all['SMILES'].apply(canonicalize_smiles)
df_all = df_all.dropna(subset=['Canonical_SMILES'])
df_clean_all = df_all.drop_duplicates(subset=['Canonical_SMILES'])
print(f"Temizlendikten sonraki EŞSİZ molekül sayısı: {len(df_clean_all)}")
df_clean_all.to_csv('leukemia_multitarget_dataset.csv', index=False)
all_smiles = df_clean_all['Canonical_SMILES'].tolist()
unique_tokens = set()

for smile in all_smiles:
    tokens = canonicalize_smiles(smile)
    unique_tokens.update(tokens)

# Özel tokenleri ekleyelim
# <PAD>: Dolgu (0), <SOS>: Başlangıç, <EOS>: Bitiş
vocab = sorted(list(unique_tokens))
vocab = ['<PAD>', '<SOS>', '<EOS>'] + vocab

# Token -> ID (stoi) ve ID -> Token (itos) sözlüklerini kuralım
stoi = { token:i for i, token in enumerate(vocab) }
itos = { i:token for i, token in enumerate(vocab) }

print(f"Toplam Benzersiz Token Sayısı (Vocab Size): {len(vocab)}")
print("Örnek Token ID'leri:", list(stoi.items())[:10])

İlaç verileri ChEMBL üzerinden çekiliyor...
  -> ABL1 taranıyor...


KeyboardInterrupt: 

In [ ]:

import torch
import numpy as np
from rdkit.Chem import QED as QEDMod

ATOM_TYPES    = ['C', 'N', 'O', 'F', 'P', 'S', 'Cl', 'Br', 'I', 'Other']
HYBRIDIZATION = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.OTHER
]
BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

NODE_FEAT_DIM = 16   # 10 atom-type + 4 hybridization + 1 H-count + 1 aromaticity
EDGE_FEAT_DIM = 4    # one-hot bond type

def atom_features(atom):
    sym      = atom.GetSymbol() if atom.GetSymbol() in ATOM_TYPES[:-1] else 'Other'
    one_hot  = [int(sym == t) for t in ATOM_TYPES]          # 10
    hyb      = [int(atom.GetHybridization() == h) for h in HYBRIDIZATION]  # 4
    h_count  = [atom.GetTotalNumHs() / 4.0]                 # 1
    aromatic = [int(atom.GetIsAromatic())]                   # 1
    return one_hot + hyb + h_count + aromatic                # 16

def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == b) for b in BOND_TYPES]

In [ ]:

from torch_geometric.data import Data

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)

    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feat  = bond_features(bond)
        edge_index += [[i, j], [j, i]]   # undirected → both directions
        edge_attr  += [feat, feat]

    if len(edge_index) == 0:             # single-atom edge case
        edge_index = [[0, 0]]
        edge_attr  = [[1, 0, 0, 0]]

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_attr,  dtype=torch.float)

    qed = torch.tensor([QEDMod.qed(mol)], dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=qed, smiles=smiles)


print("Converting SMILES to molecular graphs...")
graph_list, failed = [], 0

for smi in df_clean_all['Canonical_SMILES'].tolist():
    g = smiles_to_graph(smi)
    if g is not None:
        graph_list.append(g)
    else:
        failed += 1

print(f"Valid graphs:     {len(graph_list)}")
print(f"Failed/skipped:   {failed}")
print(f"Node feat dim:    {NODE_FEAT_DIM}")
print(f"Edge feat dim:    {EDGE_FEAT_DIM}")
print(f"Example graph:    {graph_list[0]}")


In [ ]:

from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

train_graphs, val_graphs = train_test_split(graph_list, test_size=0.1, random_state=42)

BATCH_SIZE   = 64
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_graphs,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

sample_batch = next(iter(train_loader))
print(f"Sample batch — nodes: {sample_batch.x.shape} | edges: {sample_batch.edge_index.shape}")


In [ ]:

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class MoleculeGNN(nn.Module):
    def __init__(self, node_feat_dim, hidden_dim, num_layers, dropout=0.2):
        super().__init__()

        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)

        self.convs = nn.ModuleList([
            GCNConv(hidden_dim, hidden_dim) for _ in range(num_layers)
        ])
        self.bns = nn.ModuleList([
            nn.BatchNorm1d(hidden_dim) for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.input_proj(x))

        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)

        x = global_mean_pool(x, batch)
        return self.head(x).squeeze(-1)

    def encode(self, data):
        """Return graph-level embedding (before prediction head)."""
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.input_proj(x))
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        return global_mean_pool(x, batch)   # (batch, hidden_dim)


HIDDEN_DIM = 256
NUM_LAYERS = 3      # try 4-5 for capacity test
DROPOUT    = 0.2
LR         = 1e-3
EPOCHS     = 50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = MoleculeGNN(NODE_FEAT_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nDevice: {device} | Parameters: {total_params:,}")


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = nn.MSELoss()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = batch.to(device)
            pred  = model(batch)
            loss  = criterion(pred, batch.y.squeeze())
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            n          += batch.num_graphs
    return total_loss / n

print("Eğitim Başlıyor...\n")
history = {'train': [], 'val': []}

for epoch in range(1, EPOCHS + 1):
    t0       = time.time()
    tr_loss  = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader,   train=False)
    elapsed  = time.time() - t0

    scheduler.step(val_loss)
    history['train'].append(tr_loss)
    history['val'].append(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | "
              f"Train MSE: {tr_loss:.4f} | "
              f"Val MSE: {val_loss:.4f} | "
              f"Time: {elapsed:.1f}s")

torch.save(model.state_dict(), "oncomind_gnn_model.pth")
print("\nModel kaydedildi → oncomind_gnn_model.pth")

In [ ]:

from rdkit.Chem import Descriptors, QED

def check_drug_quality(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False, "Geçersiz SMILES."
    qed_score = QED.qed(mol)
    mw        = Descriptors.MolWt(mol)
    if qed_score > 0.5 and mw < 500:
        return True, f"Uygun. QED: {qed_score:.2f}, MW: {mw:.1f}"
    return False, f"Uygunsuz. QED: {qed_score:.2f}, MW: {mw:.1f}"

example = "CC(C)(C)C(=O)N1Cc2c(NC(=O)c3cc(F)cc(F)c3)n[nH]c2C1(C)C"
is_good, reason = check_drug_quality(example)
print(reason)

In [ ]:
import torch

VOCAB_SIZE = len(vocab)
EMBED_SIZE = 128
HIDDEN_SIZE = 256
NUM_LAYERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

loaded_model = MoleculeGNN(VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS).to(device)

model_path = "oncomind_model.pth" # Dosya adın neyse
checkpoint = torch.load(model_path, map_location=device)

loaded_model.load_state_dict(checkpoint)

loaded_model.eval()